## Notebook — Sync Notes de frais Lucca → Supabase

### Source
`Lakehouse_Gold.export_compte_luccanet_gold` filtre sur les ecritures comptables de NDF :
- `libelle_ecriture LIKE 'NDF%'` OU `axe_2 = 'NDF'`
- Conventionnellement `axe_1` = code site/projet (5 chars), cle de jointure avec be_affaires

### Destination
Table `lucca_notes_frais` (Supabase)
- Cle d'idempotence : `external_id = c8_id` (id unique compta)
- `user_id` resolu via trigger DB depuis `id_lucca` (parse du libelle)

### Pattern
Edge Function `bulk-upsert`, conflict_key = `external_id`, idempotent.

In [ ]:
# ─────────────────────────────────────────
# 0) Credentials & helpers (memes que les autres notebooks)
# ─────────────────────────────────────────

# env = notebookutils.variableLibrary.getLibrary("env-lucca")
# SUPABASE_URL       = env.SUPABASE_URL
# SYNC_SECRET        = env.SYNC_SECRET
# SUPABASE_ANON_KEY  = env.SUPABASE_PUBLISHABLE_KEY

import math, time, re, requests
from decimal import Decimal
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, DateType, IntegerType

GOLD_TABLE      = "Lakehouse_Gold.export_compte_luccanet_gold"
BULK_UPSERT_URL = f"{SUPABASE_URL.rstrip('/')}/functions/v1/bulk-upsert"
BATCH_SIZE      = 500
SLEEP_S         = 0.02

def supabase_headers(json_body=True):
    h = {
        "Authorization":  f"Bearer {SUPABASE_ANON_KEY}",
        "apikey":         SUPABASE_ANON_KEY,
        "x-sync-secret":  SYNC_SECRET,
    }
    if json_body: h["Content-Type"] = "application/json"
    return h

_CTRL_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")
def _clean_str(s):
    if s is None: return None
    return _CTRL_RE.sub("", str(s)).rstrip() or None
def _to_jsonable(v):
    if v is None: return None
    if isinstance(v, bool): return v
    if isinstance(v, int): return v
    if isinstance(v, float):
        return None if (math.isnan(v) or math.isinf(v)) else round(v, 4)
    if isinstance(v, Decimal):
        try:
            f = float(v); return None if (math.isnan(f) or math.isinf(f)) else round(f, 4)
        except Exception: return _clean_str(str(v))
    if isinstance(v, (date, datetime)): return v.isoformat()
    if isinstance(v, str):
        s = _clean_str(v)
        if not s or s in ("1899-12-30", "0000-00-00"): return None
        return s
    if isinstance(v, dict): return {k: _to_jsonable(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)): return [_to_jsonable(x) for x in v]
    return _clean_str(str(v))
def _sanitize(rec): return {k: _to_jsonable(v) for k, v in rec.items()}

def _post_batch(table, conflict_key, records):
    if not records: return 0, None
    payload = {"table": table, "conflict_key": conflict_key,
               "records": [_sanitize(r) for r in records], "on_conflict": "upsert"}
    try:
        r = requests.post(BULK_UPSERT_URL, headers=supabase_headers(), json=payload, timeout=180)
        if r.status_code in (200, 201, 204): return len(records), None
        return 0, f"HTTP {r.status_code}: {r.text[:300]}"
    except Exception as e: return 0, str(e)

def send_to_supabase(table, conflict_key, records, label):
    ok_total, err_total, err_msgs = 0, 0, []
    total = len(records)
    for i in range(0, total, BATCH_SIZE):
        chunk = records[i:i + BATCH_SIZE]
        nb_ok, err = _post_batch(table, conflict_key, chunk)
        ok_total += nb_ok
        if err: err_total += len(chunk); err_msgs.append(f"batch {i//BATCH_SIZE + 1}: {err}")
        time.sleep(SLEEP_S)
    print(f"[{label}] ✅ {ok_total}/{total} envoyés | ❌ {err_total} erreurs")
    for m in err_msgs: print(f"  ⚠️  {m}")
    return ok_total, err_total

print(f"[0] Config OK · table {GOLD_TABLE}")

In [ ]:
# ─────────────────────────────────────────
# 1) Lecture + filtre NDF (libelle ou axe_2)
# ─────────────────────────────────────────

df = spark.table(GOLD_TABLE)
print(f"[1] {GOLD_TABLE} chargé : {df.count()} lignes brutes")

# Filtrer les ecritures de notes de frais : libelle commence par 'NDF' OU axe_2 = 'NDF'
df_ndf = df.filter(
    (F.upper(F.trim(F.col("libelle_ecriture"))).startswith("NDF")) |
    (F.upper(F.trim(F.col("axe_2"))) == "NDF")
)

# Garder seulement les lignes avec montant non nul et axe_1 (code projet)
df_ndf = (
    df_ndf
    .filter(F.col("solde").isNotNull() & (F.col("solde") != 0))
    .filter(F.col("axe_1").isNotNull() & (F.trim(F.col("axe_1")) != ""))
)

nb = df_ndf.count()
print(f"[1] Lignes NDF filtrees : {nb}")
df_ndf.select("c8_id", "date", "libelle_ecriture", "axe_1", "axe_2", "sens", "solde").show(5, truncate=False)

In [ ]:
# ─────────────────────────────────────────
# 2) Lookup id_lucca via libelle_ecriture
# Pattern attendu : 'NDF de NOM Prenom YYYYMM'
# On extrait le nom + prenom et on cherche profile.display_name correspondant
# Le user_id sera resolu cote DB par le trigger via id_lucca.
# Ici on resout d'abord le display_name -> id_lucca cote notebook.
# ─────────────────────────────────────────

url = f"{SUPABASE_URL.rstrip('/')}/rest/v1/profiles"
params = {
    "select":   "id_lucca,display_name",
    "id_lucca": "not.is.null",
    "limit":    "5000",
}
r = requests.get(url, headers=supabase_headers(json_body=False), params=params, timeout=60)
r.raise_for_status()
profiles_rows = r.json()

# Map normalized(name) -> id_lucca
def normalize_name(s):
    if not s: return None
    return re.sub(r"\s+", " ", s.strip().upper())

NAME_TO_ID_LUCCA = {}
for p in profiles_rows:
    if p.get("id_lucca") and p.get("display_name"):
        try:
            key = normalize_name(p["display_name"])
            NAME_TO_ID_LUCCA[key] = int(str(p["id_lucca"]).strip())
        except Exception:
            continue

print(f"[2] Profils Lucca dispo : {len(NAME_TO_ID_LUCCA)}")

# Regex pour extraire le nom : 'NDF de XXXXX YYYYYY 202XYZ' -> XXXX YYYYY
NAME_RE = re.compile(r"^NDF\s+de\s+(.+?)\s+\d{6}\s*$", re.IGNORECASE)

def parse_name_from_libelle(lib):
    if not lib: return None
    m = NAME_RE.match(lib.strip())
    if m:
        return m.group(1).strip()
    # Fallback : tout ce qui suit 'NDF de ' jusqu'a la fin (sans le YYYYMM)
    if lib.upper().startswith("NDF DE "):
        rest = lib[7:].strip()
        # remove trailing 6-digit period if present
        rest = re.sub(r"\s+\d{6}\s*$", "", rest)
        return rest or None
    return None

In [ ]:
# ─────────────────────────────────────────
# 3) Mapping vers le schema lucca_notes_frais
# Convention : montant_ht signe = solde si sens=1 (debit), -solde si sens=2 (credit)
# Pour les NDF, on attend des debits (sens=1) -> montant positif (charge)
# ─────────────────────────────────────────

rows = df_ndf.select(
    F.col("c8_id").cast(StringType()).alias("c8_id"),
    F.col("dos").cast(StringType()).alias("dos"),
    F.col("annee").cast(IntegerType()).alias("annee"),
    F.col("numero").cast(StringType()).alias("numero"),
    F.col("date").cast(DateType()).alias("date_depense"),
    F.col("journal").cast(StringType()).alias("journal"),
    F.col("compte").cast(StringType()).alias("compte_general"),
    F.col("categorie").cast(StringType()).alias("categorie"),
    F.trim(F.col("axe_1").cast(StringType())).alias("axe_1"),
    F.trim(F.col("axe_2").cast(StringType())).alias("axe_2"),
    F.col("sens").cast(IntegerType()).alias("sens"),
    F.col("solde").cast(DoubleType()).alias("solde"),
    F.col("libelle_ecriture").cast(StringType()).alias("libelle_ecriture"),
).collect()

records = []
matched_users = 0
for row in rows:
    d = row.asDict(recursive=True)
    libelle = d["libelle_ecriture"]
    name = parse_name_from_libelle(libelle)
    id_lucca = None
    if name:
        key = normalize_name(name)
        id_lucca = NAME_TO_ID_LUCCA.get(key)
        if id_lucca: matched_users += 1

    sens = d.get("sens")
    solde = d.get("solde") or 0
    montant_ht = solde if sens == 1 else -solde if sens == 2 else solde

    rec = {
        "external_id":             str(d["c8_id"]) if d.get("c8_id") else None,
        "c8_id":                   d.get("c8_id"),
        "dos":                     d.get("dos"),
        "annee":                   d.get("annee"),
        "numero":                  d.get("numero"),
        "date_depense":            d.get("date_depense"),
        "journal":                 d.get("journal"),
        "compte_general":          d.get("compte_general"),
        "categorie":               d.get("categorie"),
        "axe_1":                   d.get("axe_1"),
        "axe_2":                   d.get("axe_2"),
        "sens":                    sens,
        "montant_ht":              montant_ht,
        "libelle_ecriture":        libelle,
        "id_lucca":                id_lucca,
        "display_name_extracted":  name,
    }
    if rec["external_id"]:
        records.append(rec)

print(f"[3] Records prets : {len(records)}")
print(f"    Avec id_lucca resolu : {matched_users}")
print(f"    Sans id_lucca (orphelins, le user_id sera NULL en base) : {len(records) - matched_users}")

In [ ]:
# ─────────────────────────────────────────
# 4) Envoi vers Supabase (lucca_notes_frais)
# Conflict key = external_id (= c8_id)
# ─────────────────────────────────────────

ok, err = send_to_supabase(
    table        = "lucca_notes_frais",
    conflict_key = "external_id",
    records      = records,
    label        = "LUCCA_NDF",
)

print("")
print("=" * 60)
print("SYNC NOTES DE FRAIS LUCCA → SUPABASE — RÉSUMÉ")
print("=" * 60)
print(f"  Lignes NDF filtrees Spark : {nb}")
print(f"  Records prets             : {len(records)}")
print(f"  Upsertes Supabase         : {ok}")
print(f"  Erreurs envoi             : {err}")
print(f"  user_id non resolus       : {len(records) - matched_users}")
if err == 0:
    print("  ✅ Sync OK")
else:
    print("  ⚠️  Verifier les logs ci-dessus")
print("=" * 60)